In [1]:
import pandas as pd
import glob as glob
from Bio.Seq import Seq
import os
import numpy as np
from Bio import motifs


In [2]:
files =glob.glob('/data2st1/junyi/scenic/mouse/motif/singletons/*')

In [3]:
outdir = '/data2st1/junyi/scenic/mouse/motif/merged_cluster'

In [4]:
tbffile = '/data2st1/junyi/scenic/mouse/motif/motifs-v10nr_clust-nr.mgi-m0.001-o0.0.tbl'

In [5]:
TFoutdir = '/data2st1/junyi/output/atac1112/snregulation/'

In [6]:
df_important_TF = pd.read_csv(f'/data2st2/junyi/output/stg1028/CUMS_4VN/scenicwil_tfs_fdr_log2fc0/significant_TFlist.csv',header=None)

In [7]:
df_important_TF.columns=['TF']

In [8]:
df_tbffile = pd.read_csv(tbffile, sep='\t')

/tmp/ipykernel_514263/313275177.py:1: DtypeWarning: Columns (4) have mixed types. Specify dtype option on import or set low_memory=False.
  df_tbffile = pd.read_csv(tbffile, sep='\t')


In [9]:
df = df_tbffile
motif_similarity_fdr = 0.001,
orthologous_identity_threshold  = 0.0
df.rename(columns={'#motif_id':"MotifID",
                    'gene_name':"TF",
                    'motif_similarity_qvalue': "MotifSimilarityQvalue",
                    'orthologous_identity': "OrthologousIdentity",
                    'description': "Annotation" }, inplace=True)
df = df[(df["MotifSimilarityQvalue"] <= motif_similarity_fdr) &
        (df["OrthologousIdentity"] >= orthologous_identity_threshold)]


In [10]:
df_direct_annot = df[df['Annotation'] == 'gene is directly annotated']
#df_direct_annot = df_direct_annot.groupby(['MotifID'])['TF'].apply(lambda x: ', '.join(list(set(x)))).reset_index()


In [11]:
outdir

'/data2st1/junyi/scenic/mouse/motif/merged_cluster'

In [12]:
df_direct_annot.to_csv(os.path.join(outdir, 'motif_TF_annotated.tsv'), index=False)

In [13]:
df_direct_annot['soure_name'] = df_direct_annot['source_name']+"__"+df_direct_annot['motif_name']

/tmp/ipykernel_514263/605102179.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_direct_annot['soure_name'] = df_direct_annot['source_name']+"__"+df_direct_annot['motif_name']


In [14]:
df_direct_annot

,MotifID,motif_name,motif_description,source_name,source_version,TF,MotifSimilarityQvalue,similar_motif_id,similar_motif_description,OrthologousIdentity,orthologous_gene_name,orthologous_species,Annotation,soure_name
183,c2h2_zfs__M0369,M0369,ENSMUSG00000000317,c2h2_zfs,0.90,Bcl6b,0.0,None,None,1.0,None,None,gene is directly annotated,c2h2_zfs__M0369
184,c2h2_zfs__M0373,M0373,ENSMUSG00000022228,c2h2_zfs,0.90,Zscan26,0.0,None,None,1.0,None,None,gene is directly annotated,c2h2_zfs__M0373
185,c2h2_zfs__M0385,M0385,ENSMUSG00000028890,c2h2_zfs,0.90,Mtf1,0.0,None,None,1.0,None,None,gene is directly annotated,c2h2_zfs__M0385
186,c2h2_zfs__M0393,M0393,ENSMUSG00000033863,c2h2_zfs,0.90,Klf9,0.0,None,None,1.0,None,None,gene is directly annotated,c2h2_zfs__M0393
189,c2h2_zfs__M0400,M0400,ENSMUSG00000041703,c2h2_zfs,0.90,Zic5,0.0,None,None,1.0,None,None,gene is directly annotated,c2h2_zfs__M0400
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251819,transfac_public__M00532,M00532,V$RP58_01: RP58,transfac_public,7.0,Zbtb18,0.0,None,None,1.0,None,None,gene is directly annotated,transfac_public__M00532
251825,metacluster_71.11,M00538,V$HTF_01: HTF (XBP1),transfac_public,7.0,Xbp1,0.0,None,None,1.0,None,None,gene is directly annotated,transfac_public__M00538
251826,metacluster_57.3,M00539,V$ARNT_02: Arnt,transfac_public,7.0,Arnt,0.0,None,None,1.0,None,None,gene is directly annotated,transfac_public__M00539
251904,metacluster_198.2,M00615,V$MYCMAX_03: c-Myc:Max,transfac_public,7.0,Max,0.0,None,None,1.0,None,None,gene is directly annotated,transfac_public__M00615


In [15]:
dict_id2tf = dict(zip(df_direct_annot['soure_name'], df_direct_annot['TF']))

In [16]:
motif_similarity_annot = df[df['Annotation'].str.contains('similar') & ~df['Annotation'].str.contains('orthologous')]
#motif_similarity_annot = motif_similarity_annot.groupby(['MotifID'])['TF'].apply(lambda x: ', '.join(list(set(x)))).reset_index()


In [17]:
direct_motif_TFs = df_direct_annot[df_direct_annot.TF.isin(df_important_TF.TF)]

In [18]:
direct_motif_TFs = direct_motif_TFs.groupby(['MotifID'])['TF'].apply(lambda x: ', '.join(list(set(x)))).reset_index()


In [19]:
# direct_motifs = []
# names = []
# parsed_TFs= []
# for f in files:
#     bn = os.path.basename(f).removesuffix('.cb')
#     if bn not in direct_motif_TFs['MotifID'].values:
#         #print(f"Skipping {bn} as it is not in the important TF list.")
#         continue
#     tf_tmp = direct_motif_TFs[direct_motif_TFs['MotifID'] == bn].TF.values[0]
#     try:
#         with open(f) as handle:
#             m = motifs.read(handle, "ClusterBuster")
#             direct_motifs.append(m)
#             names.append(bn)
#             parsed_TFs.append(tf_tmp)
#             if 'MA0859.1' in record.name:
#                 print("here")

#     except ValueError:
#         with open(f) as handle:
#             records = motifs.parse(handle, "ClusterBuster")
#             for record in records:
#                 if 'MA0859.1' in record.name:
#                     print("here")
#                 record.name = bn+"#"+record.name
#                 direct_motifs.append(record)
#                 names.append(record.name)
#                 parsed_TFs.append(tf_tmp)

# mapdict = dict(zip(names, parsed_TFs))

In [20]:
extended_motif_TFs = motif_similarity_annot[motif_similarity_annot.TF.isin(df_important_TF.TF)]
extended_motif_TFs = extended_motif_TFs.groupby(['MotifID'])['TF'].apply(lambda x: ', '.join(list(set(x)))).reset_index()

In [21]:
TOBIAS_Rsults = glob.glob(f'/data2st2/junyi/output/atac1112/tobiasbam/*/*.bw/bindetect_results.xlsx')

In [23]:
# ignore cell outputs for this cell
results = pd.DataFrame()
full = pd.DataFrame()
full_results = pd.DataFrame()
for file in TOBIAS_Rsults:
    print(file)
    df_tobias = pd.read_excel(file)
    ctname = file.split('/')[-2].replace('_MC_footprints.bw','')
    region = ctname[:3]
    df_tobias['ctname'] = ctname
    df_tobias['MotifID'] = df_tobias['name'].str.split('#').str[0]
    df_tobias['source_name'] = df_tobias['name'].str.split('#').str[1]
    df_tobias['TF']=df_tobias['source_name'].map(dict_id2tf)
    df_tobias.dropna(inplace=True)
    df_tobias['abs_change'] = df_tobias['MC_MW_change'].abs()
    df_tobias.sort_values(by='abs_change', ascending=False, inplace=True)
    #df_tobias.drop_duplicates(subset=['TFID','TF'], inplace=True)
    # df_tobias["TFexpand"] = df_tobias["TF"].str.split(",\s*")  # split by comma + optional space
    # df_expanded = df_tobias.explode("TFexpand").reset_index(drop=True)
    df_expanded = df_tobias.copy()
    df_expanded['TFexpand'] = df_expanded['TF']

    # df_expanded
    df_expanded_mean = df_expanded.groupby('TFexpand').mean()
    df_expanded_mean['TF'] = df_expanded_mean.index
    df_expanded_mean['ctname'] = ctname
    df_expanded_mean['Region'] = region

    full_results = pd.concat([full_results, df_tobias], axis=0)
    results = pd.concat([results, df_expanded_mean], axis=0)

/data2st2/junyi/output/atac1112/tobiasbam/PFC/PFC_PFC_Sst_Chodl_GABA_MC_footprints.bw/bindetect_results.xlsx


/tmp/ipykernel_514263/1394294246.py:24: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df_expanded_mean = df_expanded.groupby('TFexpand').mean()


/data2st2/junyi/output/atac1112/tobiasbam/PFC/PFC_MOL-1_MC_footprints.bw/bindetect_results.xlsx


/tmp/ipykernel_514263/1394294246.py:24: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df_expanded_mean = df_expanded.groupby('TFexpand').mean()


/data2st2/junyi/output/atac1112/tobiasbam/PFC/PFC_PFC_L5_NP_Glut_MC_footprints.bw/bindetect_results.xlsx


...

/tmp/ipykernel_514263/1394294246.py:24: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df_expanded_mean = df_expanded.groupby('TFexpand').mean()


/data2st2/junyi/output/atac1112/tobiasbam/AMY/AMY_AMY_Piezo2_Cpa6_Glut_MC_footprints.bw/bindetect_results.xlsx


/tmp/ipykernel_514263/1394294246.py:24: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df_expanded_mean = df_expanded.groupby('TFexpand').mean()


/data2st2/junyi/output/atac1112/tobiasbam/AMY/AMY_MOL-2_MC_footprints.bw/bindetect_results.xlsx


/tmp/ipykernel_514263/1394294246.py:24: FutureWarning: The default value of numeric_only in DataFrameGroupBy.mean is deprecated. In a future version, numeric_only will default to False. Either specify numeric_only or select only columns which should be valid for the function.
  df_expanded_mean = df_expanded.groupby('TFexpand').mean()


In [24]:
results['Direction'] = results['MC_MW_change'].apply(lambda x: 'Up' if x > 0 else 'Down')
results['Sex'] = 'M'
results['Gene'] = results['TF']
results['ctname'] = results['ctname'].str.replace('HIP','HPF')
results["Neurotransmitter"] =  results["ctname"].apply(lambda x: 'Glut' if 'Glut' in x else ('GABA' if 'GABA' in x else 'NN'))
results['Region Subclass'] = results['ctname']
results['log2FC'] = results['MC_MW_change']
results['FDR'] = results['MC_MW_pvalue']
results['Subclass'] = results['Region Subclass'].str[4:].replace('HIP','HPF')
results['Region'] = results['Region'].str.replace('HIP','HPF')
results['status'] = results['Direction']
results['celltype.L2'] = results['Subclass']

In [25]:
full_results['Direction'] = full_results['MC_MW_change'].apply(lambda x: 'Up' if x > 0 else 'Down')
full_results['Sex'] = 'M'
full_results['Gene'] = full_results['TF']
full_results['ctname'] = full_results['ctname'].str.replace('HIP','HPF')
full_results["Neurotransmitter"] =  full_results["ctname"].apply(lambda x: 'Glut' if 'Glut' in x else ('GABA' if 'GABA' in x else 'NN'))
full_results['Region Subclass'] = full_results['ctname']
full_results['log2FC'] = full_results['MC_MW_change']
full_results['FDR'] = full_results['MC_MW_pvalue']
full_results['Subclass'] = full_results['Region Subclass'].str[4:].replace('HIP','HPF')
full_results['Region'] = full_results['Region Subclass'].str[:3].replace('HIP','HPF')
full_results['Region'] = full_results['Region'].str.replace('HIP','HPF')
full_results['status'] = full_results['Direction']
full_results['celltype.L2'] = full_results['Subclass']

In [26]:
full_results_filtered = full_results[full_results['MC_MW_highlighted']==True]

In [ ]:
full_results_filtered

In [43]:
df_regions = pd.DataFrame()
valid_files = []
unvalid_files = []
for idx, row in full_results_filtered.iterrows():
    prefix = row['output_prefix']
    region = row['Region'].replace('HPF','HIP')
    subclass = row['Subclass']
    upbed = f'/data2st2/junyi/output/atac1112/tobiasbam/{region}/{region}_{subclass}_MC_footprints.bw/{prefix}/beds/{prefix}_MC_bound.bed'
    downbed = f'/data2st2/junyi/output/atac1112/tobiasbam/{region}/{region}_{subclass}_MC_footprints.bw/{prefix}/beds/{prefix}_MW_bound.bed'

    # df_t = pd.read_csv(upbed, sep='\t')
    # df_t['TF'] = row['TF']
    # df_t['Region Subclass'] = row['Region Subclass']
    # df_regions = pd.concat([df_regions, df_t], axis=0)
    if os.path.exists(upbed):
        valid_files.append(upbed)
    else:
        unvalid_files.append(upbed)
    if os.path.exists(downbed):
        valid_files.append(downbed)
    else:
        unvalid_files.append(downbed)

In [45]:
import dask.dataframe as dd
df = dd.read_csv(valid_files, sep='\t', header=None,include_path_column="source_file")

In [49]:
df.columns = ['chr','start','end','name','score','strand','peak_chr','peak_start','peak_end','bound','source_file']

In [52]:
df.head()

,chr,start,end,name,score,strand,peak_chr,peak_start,peak_end,bound,source_file
0,chr1,6214606,6214616,metacluster_157.2homer__RATGASTCAT_JunB_None,8.35895,-,chr1,6214368,6214869,7.87081,/data2st2/junyi/output/atac1112/tobiasbam/PFC/...
1,chr1,7088711,7088721,metacluster_157.2homer__RATGASTCAT_JunB_None,8.35895,-,chr1,7088610,7089111,4.42900,/data2st2/junyi/output/atac1112/tobiasbam/PFC/...
2,chr1,13147133,13147143,metacluster_157.2homer__RATGASTCAT_JunB_None,8.35895,+,chr1,13146889,13147390,4.65615,/data2st2/junyi/output/atac1112/tobiasbam/PFC/...
3,chr1,13577101,13577111,metacluster_157.2homer__RATGASTCAT_JunB_None,8.50932,+,chr1,13576859,13577360,6.54634,/data2st2/junyi/output/atac1112/tobiasbam/PFC/...
4,chr1,13660183,13660193,metacluster_157.2homer__RATGASTCAT_JunB_None,7.91364,-,chr1,13659696,13660197,4.43627,/data2st2/junyi/output/atac1112/tobiasbam/PFC/...


In [ ]:
df_regions= df.compute()

In [54]:
df_regions

,chr,start,end,name,score,strand,peak_chr,peak_start,peak_end,bound,source_file
0,chr1,6214606,6214616,metacluster_157.2homer__RATGASTCAT_JunB_None,8.35895,-,chr1,6214368,6214869,7.87081,/data2st2/junyi/output/atac1112/tobiasbam/PFC/...
1,chr1,7088711,7088721,metacluster_157.2homer__RATGASTCAT_JunB_None,8.35895,-,chr1,7088610,7089111,4.42900,/data2st2/junyi/output/atac1112/tobiasbam/PFC/...
2,chr1,13147133,13147143,metacluster_157.2homer__RATGASTCAT_JunB_None,8.35895,+,chr1,13146889,13147390,4.65615,/data2st2/junyi/output/atac1112/tobiasbam/PFC/...
3,chr1,13577101,13577111,metacluster_157.2homer__RATGASTCAT_JunB_None,8.50932,+,chr1,13576859,13577360,6.54634,/data2st2/junyi/output/atac1112/tobiasbam/PFC/...
4,chr1,13660183,13660193,metacluster_157.2homer__RATGASTCAT_JunB_None,7.91364,-,chr1,13659696,13660197,4.43627,/data2st2/junyi/output/atac1112/tobiasbam/PFC/...
...,...,...,...,...,...,...,...,...,...,...,...
37196,chrX,167382764,167382782,metacluster_131.7hocomoco__EGR2_MOUSE.H11MO.1....,7.11719,+,chrX,167382509,167383010,1.08836,/data2st2/junyi/output/atac1112/tobiasbam/AMY/...
37197,chrX,167382807,167382825,metacluster_131.7hocomoco__EGR2_MOUSE.H11MO.1....,7.28638,-,chrX,167382509,167383010,0.89005,/data2st2/junyi/output/atac1112/tobiasbam/AMY/...
37198,chrX,168673877,168673895,metacluster_131.7hocomoco__EGR2_MOUSE.H11MO.1....,8.32336,+,chrX,168673672,168674173,2.03966,/data2st2/junyi/output/atac1112/tobiasbam/AMY/...
37199,chrX,168673958,168673976,metacluster_131.7hocomoco__EGR2_MOUSE.H11MO.1....,7.32045,-,chrX,168673672,168674173,1.95148,/data2st2/junyi/output/atac1112/tobiasbam/AMY/...


In [ ]:
results=results[results.FDR <0.05]

In [ ]:
results.to_csv(f'/data2st1/junyi/output/atac1112/tobias/tobias_TF_activity_summary_significant_TFs.csv')

In [ ]:
results['nlogp'] = -np.log10(results['FDR'] + 1e-300) * np.sign(results['log2FC'])

In [ ]:
results['Regulation'] = results['MC_MW_change'].apply(lambda x: 1 if x > 0 else -1)


In [ ]:
results['Regulation']=results['Regulation'].astype(float)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import PyComplexHeatmap as pch
from matplotlib.colors import LinearSegmentedColormap


# ---------------------------------------------------------
# 你的颜色
# ---------------------------------------------------------
region_colors = {
    "HPF": "#E13127",
    "AMY": "#6DA1D5",
    "PFC": "#F57E20",
}

nt_colors = {
    "Glut": "#FFC000",
    "GABA": "#00B050",
    "NN":   "#8c564b",
}


# ---------------------------------------------------------
# 主函数：绘制 TF × Region Subclass 热图
# ---------------------------------------------------------
def draw_tf_heatmap(df,value_col="Regulation", clipping=10,row_order=None, col_order=None):
    df = df.copy()
    if clipping:
        df[value_col] = df[value_col].clip(-1*clipping,clipping)  

    if col_order is not None:
    # Only keep columns that exist
        col_order_filtered = [c for c in col_order if c in mat.columns]
        mat = mat[col_order_filtered]

    # Apply custom TF/row order if provided
    if row_order is not None:
        row_order_filtered = [r for r in row_order if r in mat.index]
        mat = mat.loc[row_order_filtered]

    # ============================
    # 1. 构建矩阵
    # ============================
    mat = df.pivot_table(
        index="TF",
        columns="Region Subclass",
        values=value_col,
        aggfunc="mean"
    ).fillna(0)
    print("Checking columns...")

    mat = mat.replace([np.inf, -np.inf], np.nan).fillna(0)
    mat = mat.astype(float)
    mat = mat.replace([np.inf, -np.inf], np.nan)
    mat = mat.fillna(0)
    mat = mat.apply(pd.to_numeric, errors="coerce").fillna(0)


    mat = df.pivot_table(
    index="TF",
    columns="Region Subclass",
    values=value_col,
    aggfunc="mean"
    )   

    mat = mat.apply(pd.to_numeric, errors="coerce").fillna(0)

    # 1. Remove constant columns
    constant_cols = mat.columns[mat.apply(lambda x: x.nunique() <= 1)]
    print("Constant columns:", constant_cols.tolist())
    mat = mat.drop(columns=constant_cols)

    # # 2. Remove duplicate columns
    # mat_T = mat.T.drop_duplicates().T
    # removed_duplicates = set(mat.columns) - set(mat_T.columns)
    # print("Duplicate columns removed:", removed_duplicates)
    # mat = mat_T

    # 3. Final clean
    mat = mat.astype(float)
    assert np.isfinite(mat.values).all()
    # ============================
    # 2. 列注释 (Region + Neurotransmitter)
    # ============================
    col_meta = (
        df[["Region Subclass", "Region", "Neurotransmitter"]]
        .drop_duplicates()
        .set_index("Region Subclass")
        .loc[mat.columns]
    )

    # 按 region 排序列
    col_meta = col_meta.sort_values("Region")
    mat = mat[col_meta.index]

    region = col_meta["Region"]
    nt = col_meta["Neurotransmitter"]

    # ============================
    # 3. 行注释（可选：也可以不加）
    # ============================
    if "Category" in df.columns:
        row_meta = (
            df[["TF", "Category"]]
            .drop_duplicates()
            .set_index("TF")
            .loc[mat.index]
        )
        category = row_meta["Category"]
    else:
        category = None

    # ============================
    # 4. 创建顶部列注释
    # ============================
    col_ha = pch.HeatmapAnnotation(
        Region=pch.anno_simple(region, colors=region_colors, add_text=True),
        Neurotransmitter=pch.anno_simple(nt, colors=nt_colors, add_text=False),
        axis=1
    )

    # ============================
    # 5. 如果提供 Category，则加入左侧注释
    # ============================
    if category is not None:
        row_ha = pch.HeatmapAnnotation(
            Category=pch.anno_simple(category, add_text=False),
            axis=0
        )
    else:
        row_ha = None

    # ============================
    # 6. 颜色映射：蓝→白→橙，中心为0
    # ============================
    cmap = LinearSegmentedColormap.from_list(
        "blue_white_orange",
        ["#3B4CC0", "white", "#EE6A24"]
    )

    vmin = np.nanmin(mat.values)
    vmax = np.nanmax(mat.values)
    plt.figure(figsize=(10, 20))
    
    # print("Any NaN in columns? ", mat.isna().any())
    # print("Any inf? ", np.isinf(mat).any())

    # # 找出哪个 column 有 NaN / inf
    # print("Columns with NaN: ", mat.columns[mat.isna().any()].tolist())
    # print("Columns with inf: ", mat.columns[np.isinf(mat).any()].tolist())
    # ============================
    # 7. 绘制 Complex Heatmap
    # ============================
    cm = pch.ClusterMapPlotter(
        data=mat,
        top_annotation=col_ha,
        left_annotation=row_ha,
        col_split=region,      # 按 Region 分面
        row_cluster=True,
        col_cluster=True,
        row_dendrogram=True,
        col_dendrogram=True,
        cmap=cmap,
        col_split_order = ['AMY','PFC','HPF'],
        center=0,
        # col_order=col_order,        # pass user-defined order
        # row_order=row_order,        # pass user-defined order
        # col_cluster=False if col_order is not None else True,
        # row_cluster=False if row_order is not None else True,

        show_rownames=True,
        show_colnames=True,
        label="# count",
        rasterized=True
    )

    # # # 调整字体
    # if cm.ax_col_names is not None:
    #     for txt in cm.ax_col_names.texts:
    #         txt.set_rotation(90)
    #         txt.set_fontsize(7)

    # for txt in cm.ax_heatmap.get_yticklabels():
    #     txt.set_fontsize(6)

    # # 调整列名字体
    # for txt in cm.ax_heatmap.get_xticklabels():
    #     txt.set_fontsize(7)
    #     txt.set_rotation(90)


    plt.show()
    return cm
cm = draw_tf_heatmap(results[results.Neurotransmitter!='NN'],value_col="log2FC", clipping=30)

In [ ]:
cm.data2d

In [ ]:
row_names = list(cm.row_order)[0]
col_names = list(cm.col_order)

In [ ]:
cm.heatmap_axes

In [ ]:
cm = draw_tf_heatmap(results[results.Neurotransmitter=='NN'],value_col="log2FC")

In [ ]:
df_all_filtered_CUSUS = pd.read_csv('/data2st2/junyi/output/stg1028/CUMS_4VN/scenicwil_tfs_fdr_log2fc0/scenicwil_tfs_fdr_log2fc0.csv')
df_3R = df_all_filtered_CUSUS[df_all_filtered_CUSUS['Region'].isin(['HPF','AMY','PFC'])]
df_3R['Region Subclass'] = df_3R['Region subclass']
df_3R['nlogp'] = -np.log10(df_3R['FDR'] + 1e-300) * np.sign(df_3R['log2FC'])
df_3R['Neurotransmitter'] = df_3R['Region Subclass'].apply(lambda x: 'Glut' if 'Glut' in x else ('GABA' if 'GABA' in x else 'NN'))
df_3R['TF']= df_3R['TF'].str[:-3]

In [ ]:
def draw_tf_heatmap_ordered(df,value_col="Regulation",row='TF',column='Region Subclass',
                            col_meta=['Region Subclass', 'Region', 'Neurotransmitter'],
                            clipping=10,row_order=None, col_order=None):
    df = df.copy()
    if clipping:
        df[value_col] = df[value_col].clip(-1*clipping,clipping)  
    
    if row_order is not None:
        df = df[df[row].isin(row_order)]
    if col_order is not None:
        df = df[df[column].isin(col_order)]
    # ============================
    # 1. 构建矩阵
    # ============================
    mat = df.pivot_table(
        index=row,
        columns=column,
        values=value_col,
        aggfunc="mean"
    ).fillna(0)
    print("Checking columns...")

    mat = mat.replace([np.inf, -np.inf], np.nan).fillna(0)
    mat = mat.astype(float)
    mat = mat.replace([np.inf, -np.inf], np.nan)
    mat = mat.fillna(0)
    mat = mat.apply(pd.to_numeric, errors="coerce").fillna(0)

    constant_cols = mat.columns[mat.apply(lambda x: x.nunique() <= 1)]
    print("Constant columns:", constant_cols.tolist())
    mat = mat.drop(columns=constant_cols)

    # # 2. Remove duplicate columns
    # mat_T = mat.T.drop_duplicates().T
    # removed_duplicates = set(mat.columns) - set(mat_T.columns)
    # print("Duplicate columns removed:", removed_duplicates)
    # mat = mat_T

    # 3. Final clean
    mat = mat.astype(float)
    assert np.isfinite(mat.values).all()

    # order the mat if row_order and col_order are provided
    if col_order is not None:
        # Only keep columns that exist
        col_order_filtered = [c for c in col_order if c in mat.columns]
        mat = mat[col_order_filtered]
    if row_order is not None:
        row_order_filtered = [r for r in row_order if r in mat.index]
        mat = mat.loc[row_order_filtered]
    # ============================
    # 2. 列注释 (Region + Neurotransmitter)
    # ============================
    col_meta = (
        df[["Region Subclass", "Region", "Neurotransmitter"]]
        .drop_duplicates()
        .set_index("Region Subclass")
        .loc[mat.columns]
    )

    # 按 region 排序列
    col_meta = col_meta.sort_values("Region")
    mat = mat[col_meta.index]
    region = col_meta["Region"]
    nt = col_meta["Neurotransmitter"]

    # ============================
    # 3. 行注释（可选：也可以不加）
    # ============================
    if "Category" in df.columns:
        row_meta = (
            df[["TF", "Category"]]
            .drop_duplicates()
            .set_index("TF")
            .loc[mat.index]
        )
        category = row_meta["Category"]
    else:
        category = None

    # ============================
    # 4. 创建顶部列注释
    # ============================
    col_ha = pch.HeatmapAnnotation(
        Region=pch.anno_simple(region, colors=region_colors, add_text=True),
        Neurotransmitter=pch.anno_simple(nt, colors=nt_colors, add_text=False),
        axis=1
    )
    # ============================
    # 5. 如果提供 Category，则加入左侧注释
    # ============================
    if category is not None:
        row_ha = pch.HeatmapAnnotation(
            Category=pch.anno_simple(category, add_text=False),
            axis=0
        )
    else:
        row_ha = None

    # ============================
    # 6. 颜色映射：蓝→白→橙，中心为0
    # ============================
    cmap = LinearSegmentedColormap.from_list(
        "blue_white_orange",
        ["#3B4CC0", "white", "#EE6A24"]
    )
    plt.figure(figsize=(10, 20))

    row_cluster = True if row_order is None else False
    col_cluster = True if col_order is None else False
    row_dendrogram = True if row_order is None else False
    col_dendrogram = True if col_order is None else False

    cm = pch.ClusterMapPlotter(
        data=mat,
        top_annotation=col_ha,
        left_annotation=row_ha,
        col_split=region,      # 按 Region 分面
        row_cluster=row_cluster,
        col_cluster=col_cluster,
        row_dendrogram=row_dendrogram,
        col_dendrogram=col_dendrogram,
        cmap=cmap,
        col_split_order = ['AMY','PFC','HPF'],
        center=0,
        show_rownames=True,
        show_colnames=True,
        label="# count",
        rasterized=True
    )
    plt.show()
    return cm

cm2 = draw_tf_heatmap_ordered(df_3R[df_3R.Neurotransmitter!='NN'],row_order=row_names,value_col="nlogp", clipping=50)

In [ ]:
df_3R

In [ ]:
cm = draw_tf_heatmap(df_3R[df_3R.Neurotransmitter!='NN'], value_col="nlogp", clipping=1)

In [ ]:
df_tobias_drop = df_tobias.sort_values(by='abs_change',ascending=False).drop_duplicates(subset=['TFID'], keep='first')

In [ ]:
df_tobias_drop = df_tobias_drop[~df_tobias_drop['name'].str.contains('meta')]

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# df = your TOBIAS BINDetect table
df_tobias_drop["neglog10_p"] = -np.log10(df_tobias_drop["MC_MW_pvalue"])

# ---- Top 20 UP (highest positive change) ----
top20_up = df_tobias_drop.nlargest(20, "MC_MW_change").copy()
top20_up["highlight"] = "up"

# ---- Top 20 DOWN (most negative change) ----
top20_down = df_tobias_drop.nsmallest(20, "MC_MW_change").copy()
top20_down["highlight"] = "down"

# ---- Combine top40 ----
top40 = pd.concat([top20_up, top20_down], axis=0)

# ---- Volcano plot ----
plt.figure(figsize=(10, 5))

# All background points (gray)
sns.scatterplot(
    data=df_tobias_drop,
    x="MC_MW_change",
    y="neglog10_p",
    color="lightgray",
    s=40
)

# Up 20 (red)
sns.scatterplot(
    data=top20_up,
    x="MC_MW_change",
    y="neglog10_p",
    color="red",
    s=80,
    edgecolor="black"
)

# Down 20 (blue)
sns.scatterplot(
    data=top20_down,
    x="MC_MW_change",
    y="neglog10_p",
    color="blue",
    s=80,
    edgecolor="black"
)

# ---- Add labels ----
for _, row in top40.iterrows():
    plt.text(
        row["MC_MW_change"],
        row["neglog10_p"],
        row["TF"],        # Your TF column name
        fontsize=8,
        ha="right",
        va="bottom"
    )

# ---- Axis + Style ----
plt.axvline(0, color="gray", linestyle="--")
plt.xlabel("Motif accessibility change (MC_MW_change)", fontsize=14)
plt.ylabel("-log10(p-value)", fontsize=14)
plt.title("Motif Accessibility Volcano Plot (Top20 Up / Top20 Down)", fontsize=16)
